# H3: Is a route's mean arrival delay associated with distance and with origin congestion?

Pre-registered: route-months (origin, dest, year, month) with at least 30 completed flights, January 2023 to December 2025. Congestion = scheduled departures at the origin that month. Pearson r (Fisher z CI) and Spearman rho (bootstrap CI) for each pair; LOWESS scatter plots; partial correlation of delay with congestion controlling for month of year.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from delaystats import plotting, results
from delaystats import stats_helpers as sh
from delaystats.athena import run_query

ALPHA = 0.05  # pre-registered, two-sided (docs/ANALYSIS_PLAN.md)
SEED = 20261014  # fixed for every bootstrap
pd.set_option("display.float_format", "{:,.4f}".format)


def fmt_p(p):
    return "< 0.0001" if p < 1e-4 else f"{p:.4f}"


def table(rs):
    return pd.DataFrame([r.as_row() for r in rs]).set_index("test")


PLOT_SAMPLE = 5_000  # points drawn in each scatter (all rows are used in the tests)

## Data

In [ ]:
rm = run_query("h3_route_month")
print(f"{len(rm):,} route-months, {rm[['origin', 'dest']].drop_duplicates().shape[0]:,} routes")
rm[["mean_arrdelay", "distance", "scheduled_departures", "flights"]].describe()

## Correlations

In [ ]:
y = rm["mean_arrdelay"].to_numpy(float)
dist = rm["distance"].to_numpy(float)
cong = rm["scheduled_departures"].to_numpy(float)
h3 = [
    sh.pearson_with_ci(dist, y, name="h3_pearson_distance"),
    sh.spearman_with_bootstrap_ci(dist, y, name="h3_spearman_distance", n_boot=2000, seed=SEED),
    sh.pearson_with_ci(cong, y, name="h3_pearson_congestion"),
    sh.spearman_with_bootstrap_ci(cong, y, name="h3_spearman_congestion", n_boot=2000, seed=SEED),
]

months = pd.get_dummies(rm["month"], prefix="m", drop_first=True).astype(float)
data = pd.concat([rm[["mean_arrdelay", "scheduled_departures"]], months], axis=1)
h3.append(
    sh.partial_corr(
        data,
        "scheduled_departures",
        "mean_arrdelay",
        list(months.columns),
        name="h3_partial_congestion_given_month",
    )
)
results.save("H3", h3)
table(h3)

## Scatter plots

In [ ]:
sample = rm.sample(n=min(PLOT_SAMPLE, len(rm)), random_state=SEED)
plotting.scatter_with_lowess(
    sample["distance"].to_numpy(),
    sample["mean_arrdelay"].to_numpy(),
    "h3_delay_vs_distance",
    f"H3: route-month mean delay vs distance ({len(sample):,} sampled points)",
    "Distance (miles)",
    "Mean arrival delay (minutes)",
)
plotting.scatter_with_lowess(
    sample["scheduled_departures"].to_numpy(),
    sample["mean_arrdelay"].to_numpy(),
    "h3_delay_vs_congestion",
    f"H3: route-month mean delay vs origin departures ({len(sample):,} sampled points)",
    "Scheduled departures at origin that month (log scale)",
    "Mean arrival delay (minutes)",
    log_x=True,
)

## Interpretation

In [ ]:
pd_, sd_, pc_, sc_, part = h3


def strength(r):
    r = abs(r)
    return "negligible" if r < 0.1 else "weak" if r < 0.3 else "moderate" if r < 0.5 else "strong"


def line(p, s, label):
    gap = abs(p.effect_size - s.effect_size)
    why = (
        "Pearson and Spearman agree, so the relationship is roughly linear and not driven by a few extreme route-months."
        if gap < 0.05
        else "The gap between Pearson and Spearman points to outliers or a non-linear (monotonic but curved) relationship; Spearman, being rank-based, is the steadier summary."
    )
    return (
        f"- **{label}:** Pearson r = {p.effect_size:.3f} (95% CI [{p.effect_ci[0]:.3f}, {p.effect_ci[1]:.3f}], "
        f"p = {fmt_p(p.p)}), Spearman rho = {s.effect_size:.3f} (95% CI [{s.effect_ci[0]:.3f}, {s.effect_ci[1]:.3f}]): "
        f"a {strength(s.effect_size)} association. {why}"
    )


display(
    Markdown(f"""
Across {len(rm):,} route-months (alpha = {ALPHA}, two-sided; five pre-registered estimates, reported
with confidence intervals rather than corrected p-values, because every one is significant at this n
and the size of r is what carries meaning):

{line(pd_, sd_, "Distance")}
{line(pc_, sc_, "Origin congestion")}
- **Controlling for month of year:** the partial correlation of congestion with delay is
  {part.effect_size:.3f} (95% CI [{part.effect_ci[0]:.3f}, {part.effect_ci[1]:.3f}]), against {pc_.effect_size:.3f}
  without the adjustment; seasonality {"accounts for little of" if abs(part.effect_size - pc_.effect_size) < 0.02 else "changes"} the
  congestion association.
- **Not causal.** Correlation does not show that congestion causes delay. Two plausible confounders:
  **hub status** (big hubs schedule more departures and also connect more late-arriving aircraft) and
  **weather exposure** (some airports are busy and weather-prone at the same time).
- Survivorship: route-months with fewer than 30 flights are excluded, so thin routes are not represented.
""")
)